In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder \
    .appName("SmartEnergyMonitoring") \
    .getOrCreate()

In [0]:
sensor_data = [

(101,"AC Unit","Conference Room","2026-06-01 09:00:00",8.5),
(101,"AC Unit","Conference Room","2026-06-01 18:00:00",7.2),
(102,"Server Rack","Server Room","2026-06-01 10:00:00",25.5),
(103,"LED Lights","Lab","2026-06-01 12:00:00",4.0),
(101,"AC Unit","Conference Room","2026-06-02 09:30:00",9.0),
(102,"Server Rack","Server Room","2026-06-02 11:00:00",24.8),
(103,"LED Lights","Lab","2026-06-02 18:30:00",3.5),
(104,"Projector","Conference Room","2026-06-03 14:00:00",6.5),
(105,"Printer","Office","2026-06-03 16:00:00",2.2),
(102,"Server Rack","Server Room","2026-06-04 09:15:00",26.1)

]

columns = [
    "device_id",
    "device_name",
    "room",
    "timestamp",
    "energy_kwh"
]

logs_df = spark.createDataFrame(sensor_data, columns)

display(logs_df)


device_id,device_name,room,timestamp,energy_kwh
101,AC Unit,Conference Room,2026-06-01 09:00:00,8.5
101,AC Unit,Conference Room,2026-06-01 18:00:00,7.2
102,Server Rack,Server Room,2026-06-01 10:00:00,25.5
103,LED Lights,Lab,2026-06-01 12:00:00,4.0
101,AC Unit,Conference Room,2026-06-02 09:30:00,9.0
102,Server Rack,Server Room,2026-06-02 11:00:00,24.8
103,LED Lights,Lab,2026-06-02 18:30:00,3.5
104,Projector,Conference Room,2026-06-03 14:00:00,6.5
105,Printer,Office,2026-06-03 16:00:00,2.2
102,Server Rack,Server Room,2026-06-04 09:15:00,26.1


In [0]:
logs_df.coalesce(1).write \
.mode("overwrite") \
.option("header",True) \
.csv("/FileStore/tables/sensor_logs")

In [0]:
logs = spark.read \
.option("header",True) \
.option("inferSchema",True) \
.csv("/FileStore/tables/sensor_logs")

display(logs)

device_id,device_name,room,timestamp,energy_kwh
101,AC Unit,Conference Room,2026-06-01T09:00:00.000Z,8.5
101,AC Unit,Conference Room,2026-06-01T18:00:00.000Z,7.2
102,Server Rack,Server Room,2026-06-01T10:00:00.000Z,25.5
103,LED Lights,Lab,2026-06-01T12:00:00.000Z,4.0
101,AC Unit,Conference Room,2026-06-02T09:30:00.000Z,9.0
102,Server Rack,Server Room,2026-06-02T11:00:00.000Z,24.8
103,LED Lights,Lab,2026-06-02T18:30:00.000Z,3.5
104,Projector,Conference Room,2026-06-03T14:00:00.000Z,6.5
105,Printer,Office,2026-06-03T16:00:00.000Z,2.2
102,Server Rack,Server Room,2026-06-04T09:15:00.000Z,26.1


In [0]:
logs = logs.withColumn(
    "energy_kwh",
    col("energy_kwh").cast("double")
)

logs = logs.withColumn(
    "timestamp",
    to_timestamp("timestamp")
)

logs = logs.dropna()

display(logs)

device_id,device_name,room,timestamp,energy_kwh
101,AC Unit,Conference Room,2026-06-01T09:00:00.000Z,8.5
101,AC Unit,Conference Room,2026-06-01T18:00:00.000Z,7.2
102,Server Rack,Server Room,2026-06-01T10:00:00.000Z,25.5
103,LED Lights,Lab,2026-06-01T12:00:00.000Z,4.0
101,AC Unit,Conference Room,2026-06-02T09:30:00.000Z,9.0
102,Server Rack,Server Room,2026-06-02T11:00:00.000Z,24.8
103,LED Lights,Lab,2026-06-02T18:30:00.000Z,3.5
104,Projector,Conference Room,2026-06-03T14:00:00.000Z,6.5
105,Printer,Office,2026-06-03T16:00:00.000Z,2.2
102,Server Rack,Server Room,2026-06-04T09:15:00.000Z,26.1


In [0]:

logs = logs.withColumn(
    "date",
    to_date("timestamp")
)

logs = logs.withColumn(
    "week",
    weekofyear("timestamp")
)

display(logs)

device_id,device_name,room,timestamp,energy_kwh,date,week
101,AC Unit,Conference Room,2026-06-01T09:00:00.000Z,8.5,2026-06-01,23
101,AC Unit,Conference Room,2026-06-01T18:00:00.000Z,7.2,2026-06-01,23
102,Server Rack,Server Room,2026-06-01T10:00:00.000Z,25.5,2026-06-01,23
103,LED Lights,Lab,2026-06-01T12:00:00.000Z,4.0,2026-06-01,23
101,AC Unit,Conference Room,2026-06-02T09:30:00.000Z,9.0,2026-06-02,23
102,Server Rack,Server Room,2026-06-02T11:00:00.000Z,24.8,2026-06-02,23
103,LED Lights,Lab,2026-06-02T18:30:00.000Z,3.5,2026-06-02,23
104,Projector,Conference Room,2026-06-03T14:00:00.000Z,6.5,2026-06-03,23
105,Printer,Office,2026-06-03T16:00:00.000Z,2.2,2026-06-03,23
102,Server Rack,Server Room,2026-06-04T09:15:00.000Z,26.1,2026-06-04,23


In [0]:
daily_summary = logs.groupBy(
    "date",
    "room"
).agg(
    sum("energy_kwh").alias("daily_energy"),
    avg("energy_kwh").alias("average_energy")
)

display(daily_summary)

date,room,daily_energy,average_energy
2026-06-01,Lab,4.0,4.0
2026-06-02,Conference Room,9.0,9.0
2026-06-01,Conference Room,15.7,7.85
2026-06-01,Server Room,25.5,25.5
2026-06-02,Server Room,24.8,24.8
2026-06-02,Lab,3.5,3.5
2026-06-03,Office,2.2,2.2
2026-06-03,Conference Room,6.5,6.5
2026-06-04,Server Room,26.1,26.1


In [0]:
weekly_summary = logs.groupBy(
    "week",
    "room"
).agg(
    sum("energy_kwh").alias("weekly_energy"),
    avg("energy_kwh").alias("average_energy")
)

display(weekly_summary)

week,room,weekly_energy,average_energy
23,Server Room,76.4,25.46666666666667
23,Lab,7.5,3.75
23,Office,2.2,2.2
23,Conference Room,31.2,7.8


In [0]:
alerts = daily_summary.withColumn(
    "alert",
    when(col("daily_energy") > 20, "High Usage")
    .when(col("daily_energy") > 10, "Medium Usage")
    .otherwise("Normal")
)

display(alerts)


date,room,daily_energy,average_energy,alert
2026-06-01,Lab,4.0,4.0,Normal
2026-06-02,Conference Room,9.0,9.0,Normal
2026-06-01,Conference Room,15.7,7.85,Medium Usage
2026-06-01,Server Room,25.5,25.5,High Usage
2026-06-02,Server Room,24.8,24.8,High Usage
2026-06-02,Lab,3.5,3.5,Normal
2026-06-03,Office,2.2,2.2,Normal
2026-06-03,Conference Room,6.5,6.5,Normal
2026-06-04,Server Room,26.1,26.1,High Usage


In [0]:
daily_summary.write \
.mode("overwrite") \
.format("delta") \
.save("/FileStore/delta/daily_summary")

In [0]:
spark.sql("""

CREATE TABLE IF NOT EXISTS daily_summary

USING DELTA

LOCATION '/FileStore/delta/daily_summary'

""")

spark.sql("""

CREATE TABLE IF NOT EXISTS weekly_summary

USING DELTA

LOCATION '/FileStore/delta/weekly_summary'

""")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7586744200741173>, line 1
----> 1 spark.sql("""
      2 
      3 CREATE TABLE IF NOT EXISTS daily_summary
      4 
      5 USING DELTA
      6 
      7 LOCATION '/FileStore/delta/daily_summary'
      8 
      9 """)
     11 spark.sql("""
     12 
     13 CREATE TABLE IF NOT EXISTS weekly_summary
   (...)
     18 
     19 """)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/session.py:901, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    898         _views.append(SubqueryAlias(df._plan, name))
    900 cmd = SQL(sqlQuery, _args, _named_args, _views)
--> 901 data, properties, ei = self.client.execute_command(cmd.command(self._client))
    902 if "sql_command_result" in properties:
    903     df = DataFrame(CachedRelation(properties["sql_command_result"]), self)

File /databricks/python/lib/py

In [0]:
daily_summary.coalesce(1).write \
.mode("overwrite") \
.option("header",True) \
.csv("/FileStore/output/daily_summary")

weekly_summary.coalesce(1).write \
.mode("overwrite") \
.option("header",True) \
.csv("/FileStore/output/weekly_summary")

In [0]:
daily = spark.read.format("delta").load("/FileStore/delta/daily_summary")

weekly = spark.read.format("delta").load("/FileStore/delta/weekly_summary")

display(daily)

display(weekly)

date,room,daily_energy,average_energy
2026-06-01,Lab,4.0,4.0
2026-06-02,Conference Room,9.0,9.0
2026-06-01,Conference Room,15.7,7.85
2026-06-01,Server Room,25.5,25.5
2026-06-02,Server Room,24.8,24.8
2026-06-02,Lab,3.5,3.5
2026-06-03,Office,2.2,2.2
2026-06-03,Conference Room,6.5,6.5
2026-06-04,Server Room,26.1,26.1


week,room,weekly_energy,average_energy
23,Server Room,76.4,25.46666666666667
23,Lab,7.5,3.75
23,Office,2.2,2.2
23,Conference Room,31.2,7.8


In [0]:
logs.createOrReplaceTempView("energy_logs")

spark.sql("""

SELECT
room,
SUM(energy_kwh) AS total_energy
FROM energy_logs
GROUP BY room
ORDER BY total_energy DESC

""").show()

+---------------+------------+
|           room|total_energy|
+---------------+------------+
|    Server Room|        76.4|
|Conference Room|        31.2|
|            Lab|         7.5|
|         Office|         2.2|
+---------------+------------+



In [0]:
print("Daily Summary")
daily.show()

print("Weekly Summary")
weekly.show()

print("Energy Alerts")
alerts.show()

Daily Summary
+----------+---------------+------------+--------------+
|      date|           room|daily_energy|average_energy|
+----------+---------------+------------+--------------+
|2026-06-01|            Lab|         4.0|           4.0|
|2026-06-02|Conference Room|         9.0|           9.0|
|2026-06-01|Conference Room|        15.7|          7.85|
|2026-06-01|    Server Room|        25.5|          25.5|
|2026-06-02|    Server Room|        24.8|          24.8|
|2026-06-02|            Lab|         3.5|           3.5|
|2026-06-03|         Office|         2.2|           2.2|
|2026-06-03|Conference Room|         6.5|           6.5|
|2026-06-04|    Server Room|        26.1|          26.1|
+----------+---------------+------------+--------------+

Weekly Summary
+----+---------------+-------------+-----------------+
|week|           room|weekly_energy|   average_energy|
+----+---------------+-------------+-----------------+
|  23|    Server Room|         76.4|25.46666666666667|
|  23|   